# 05 · Better tree-shade features (fast NumPy lookups)

**Run this in Google Colab.** It takes about 15–25 minutes.

Why: in notebook 03, tree cover in the ~28 m block around each roof came out as 100% for most roofs, because
the 10 m satellite map often cannot see small Kerala houses under coconut and homestead trees.

What it adds to every roof:
1. **`roof_seen_as`**: what ESA WorldCover sees at the roof's own 10 m pixel: `built-up`, `tree` or `other`.
2. **`tree_pct_140m`**: % trees in about 140 m around the roof (near neighbourhood).
3. **`tree_pct_300m`**: % trees in about 300 m around the roof (wider area).

It also removes the one roof that notebook 03 counted twice, and keeps each roof's centre point
(`centre_lon`, `centre_lat`) for maps and the dashboard.

Output: `MyDrive/rooftop_solar/data/features/roof_features_v2` (split by district).

These lookups use NumPy on whole raster arrays instead of Sedona's per-roof raster join
(which took 6.4 hours in notebook 03); the log prints the time for comparison.

**How to run:** Runtime → Run all. Allow Google Drive access.

In [ ]:
# Step 1: connect Google Drive, set folders, copy the inputs to the fast Colab disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
RAW = f'{PROJECT}/data/raw'
FEATURES = f'{PROJECT}/data/features'
RESULTS = f'{PROJECT}/results'
LOGS = f'{PROJECT}/logs'
LOCAL = '/content/work'      # fast Colab disk, deleted when the session ends
for folder in (RESULTS, LOGS, LOCAL):
    os.makedirs(folder, exist_ok=True)

LOG_FILE = f'{LOGS}/05_log.txt'
open(LOG_FILE, 'w').close()


def log(*parts):
    """Print, and also save to Google Drive so results survive if the tab closes."""
    text = ' '.join(str(p) for p in parts)
    print(text)
    with open(LOG_FILE, 'a') as f:
        f.write(text + '\n')


def log_table(pdf, csv_name=None):
    """Log a small pandas table; optionally save it as a CSV in results/."""
    log(pdf.to_string(index=False))
    if csv_name:
        pdf.to_csv(f'{RESULTS}/{csv_name}', index=False)


T_START = time.time()
shutil.copytree(f'{FEATURES}/roof_features', f'{LOCAL}/roof_features', dirs_exist_ok=True)
shutil.copy(f'{RAW}/tree_pct_kerala.tif', f'{LOCAL}/tree_pct_kerala.tif')
log(f'Inputs copied to {LOCAL} in {time.time() - T_START:.0f} s')

In [ ]:
# Step 2: install DuckDB and rasterio (NumPy, pandas and SciPy are already in Colab)
!pip install -q duckdb==2.0.0 rasterio

import urllib.request
import duckdb
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from scipy.ndimage import uniform_filter

con = duckdb.connect(f'{LOCAL}/roofs.duckdb')   # a file, so DuckDB can spill to disk if memory runs short
con.sql("INSTALL spatial; LOAD spatial;")
con.sql("SET geometry_always_xy = true; SET memory_limit = '6GB';")   # leave RAM for the NumPy steps

In [ ]:
# Step 3: load all roofs once (one row per roof id) and compute each roof's centre point
t = time.time()
con.execute(f"""
    CREATE OR REPLACE TABLE roofs AS
    SELECT row_number() OVER () AS rid, *
    FROM (
        SELECT *, ST_X(ST_Centroid(geometry)) AS centre_lon, ST_Y(ST_Centroid(geometry)) AS centre_lat
        FROM read_parquet('{LOCAL}/roof_features/*/*.parquet', hive_partitioning = true)
        QUALIFY row_number() OVER (PARTITION BY id ORDER BY tree_pct NULLS LAST) = 1
    )
""").fetchall()
ROOFS = con.execute('SELECT count(*) FROM roofs').fetchall()[0][0]
log(f'Roofs loaded (one row per id): {ROOFS:,} in {time.time() - t:.0f} s')

xy = con.execute('SELECT rid, centre_lon, centre_lat FROM roofs ORDER BY rid').fetchnumpy()
rid = np.asarray(xy['rid'])
lon = np.asarray(xy['centre_lon'], dtype='float64')
lat = np.asarray(xy['centre_lat'], dtype='float64')
extra = pd.DataFrame({'rid': rid})

In [ ]:
# Step 4: tree cover in a wider neighbourhood, from the ~28 m tree-% raster of notebook 03
t = time.time()
with rasterio.open(f'{LOCAL}/tree_pct_kerala.tif') as src:
    tree = src.read(1)
    T = src.transform
valid = tree != 255                                   # 255 = no data
vals = np.where(valid, tree, 0).astype('float32')
validf = valid.astype('float32')
rows = np.floor((lat - T.f) / T.e).astype(np.int64)
cols = np.floor((lon - T.c) / T.a).astype(np.int64)
inside = (rows >= 0) & (rows < tree.shape[0]) & (cols >= 0) & (cols < tree.shape[1])
r_in, c_in = rows[inside], cols[inside]

for size, name in ((5, 'tree_pct_140m'), (11, 'tree_pct_300m')):   # 5 x 28 m ≈ 140 m, 11 x 28 m ≈ 300 m
    num = uniform_filter(vals, size=size, mode='nearest')            # moving average of tree %
    den = uniform_filter(validf, size=size, mode='nearest')          # share of pixels with data
    out = np.full(len(rid), np.nan, dtype='float32')
    d = den[r_in, c_in]
    out[inside] = np.where(d > 0, num[r_in, c_in] / np.maximum(d, 1e-6), np.nan)
    extra[name] = np.round(out, 1)
    del num, den
del tree, valid, vals, validf
T_TREE = time.time() - t
log(f'Neighbourhood tree cover (~140 m and ~300 m) for {len(rid):,} roofs in {T_TREE:.0f} s')

In [ ]:
# Step 5: what ESA WorldCover (10 m) sees at each roof's own centre pixel (50 = built-up, 10 = tree)
WC_URL = 'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_{}_Map.tif'
TILES = ['N06E075', 'N09E072', 'N09E075', 'N12E072', 'N12E075']
PIX = 1 / 12000            # WorldCover pixel size in degrees
CHUNK = 4096               # tile rows read at a time (keeps memory low)

t = time.time()
wc_class = np.zeros(len(rid), dtype=np.uint8)          # 0 = no WorldCover value found
for tile in TILES:
    path = f'{LOCAL}/{tile}.tif'
    if not os.path.exists(path):
        urllib.request.urlretrieve(WC_URL.format(tile), path)
    with rasterio.open(path) as src:
        left, bottom, right, top = src.bounds
        sel = np.nonzero((lon >= left) & (lon < right) & (lat > bottom) & (lat <= top))[0]
        if len(sel) == 0:
            continue
        r = np.floor((top - lat[sel]) / PIX).astype(np.int64)
        c = np.floor((lon[sel] - left) / PIX).astype(np.int64)
        order = np.argsort(r, kind='stable')
        sel, r, c = sel[order], r[order], c[order]
        for b0 in range(int(r[0]) // CHUNK * CHUNK, int(r[-1]) + 1, CHUNK):
            i0, i1 = np.searchsorted(r, [b0, b0 + CHUNK])
            if i0 == i1:
                continue
            cc = c[i0:i1]
            c0, c1 = int(cc.min()), int(cc.max()) + 1
            block = src.read(1, window=Window(c0, b0, c1 - c0, min(CHUNK, src.height - b0)))
            wc_class[sel[i0:i1]] = block[r[i0:i1] - b0, cc - c0]
    log(f'  WorldCover tile {tile}: {len(sel):,} roofs')
extra['wc_class'] = wc_class
T_WC = time.time() - t
log(f'WorldCover class at roof centres in {T_WC:.0f} s')

In [ ]:
# Step 6: join the new columns to the roofs and save, split by district
t = time.time()
shutil.rmtree(f'{LOCAL}/roof_features_v2', ignore_errors=True)
con.register('extra_df', extra)
con.execute(f"""
    COPY (
        SELECT r.* EXCLUDE (rid),
               e.wc_class,
               CASE e.wc_class WHEN 50 THEN 'built-up' WHEN 10 THEN 'tree' WHEN 0 THEN NULL ELSE 'other' END
                   AS roof_seen_as,
               e.tree_pct_140m, e.tree_pct_300m
        FROM roofs r JOIN extra_df e USING (rid)
    ) TO '{LOCAL}/roof_features_v2' (FORMAT parquet, PARTITION_BY (district), COMPRESSION zstd)
""").fetchall()
log(f'roof_features_v2 written in {time.time() - t:.0f} s')

In [ ]:
# Step 7: checks and summaries (send these to Claude)
V2 = f"read_parquet('{LOCAL}/roof_features_v2/*/*.parquet', hive_partitioning = true)"

log('\nCHECKS (roofs = distinct_ids, the rest 0 is ideal)')
log_table(con.sql(f"""
    SELECT count(*) AS roofs, count(DISTINCT id) AS distinct_ids,
           sum(CAST(wc_class = 0 AS INT))         AS no_worldcover,
           sum(CAST(tree_pct_140m IS NULL AS INT)) AS missing_140m,
           sum(CAST(tree_pct_300m IS NULL AS INT)) AS missing_300m
    FROM {V2}
""").df())

log('\nWHAT THE SATELLITE SEES AT THE ROOF CENTRE')
log_table(con.sql(f"""
    SELECT roof_seen_as, count(*) AS roofs, round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM {V2} GROUP BY roof_seen_as ORDER BY roofs DESC
""").df(), 'roof_seen_as.csv')

log('\nTREE FEATURES: share of roofs at exactly 100% and 5th / 50th / 95th percentile')
log_table(con.sql(f"""
    SELECT 'tree_pct (28 m)' AS feature, round(100 * avg(CAST(tree_pct = 100 AS DOUBLE)), 1) AS pct_at_100,
           approx_quantile(tree_pct, [0.05, 0.5, 0.95]) AS p5_p50_p95 FROM {V2}
    UNION ALL
    SELECT 'tree_pct_140m', round(100 * avg(CAST(tree_pct_140m = 100 AS DOUBLE)), 1),
           approx_quantile(tree_pct_140m, [0.05, 0.5, 0.95]) FROM {V2}
    UNION ALL
    SELECT 'tree_pct_300m', round(100 * avg(CAST(tree_pct_300m = 100 AS DOUBLE)), 1),
           approx_quantile(tree_pct_300m, [0.05, 0.5, 0.95]) FROM {V2}
""").df())

log('\nBY DISTRICT')
log_table(con.sql(f"""
    SELECT district, count(*) AS roofs,
           round(100 * avg(CAST(roof_seen_as = 'built-up' AS DOUBLE)), 1) AS pct_seen_built_up,
           round(avg(tree_pct), 1)      AS avg_tree_28m,
           round(avg(tree_pct_140m), 1) AS avg_tree_140m,
           round(avg(tree_pct_300m), 1) AS avg_tree_300m
    FROM {V2} GROUP BY district ORDER BY roofs DESC
""").df(), 'district_tree_features.csv')

log(f'\nTiming: NumPy tree lookups {T_TREE:.0f} s + WorldCover lookups {T_WC:.0f} s '
    f'(Sedona raster join in notebook 03: 385.7 min)')

In [ ]:
# Step 8: copy the new roof features to Google Drive
t = time.time()
shutil.rmtree(f'{FEATURES}/roof_features_v2', ignore_errors=True)    # remove an older copy first
shutil.copytree(f'{LOCAL}/roof_features_v2', f'{FEATURES}/roof_features_v2')
log(f'Saved {FEATURES}/roof_features_v2 in {time.time() - t:.0f} s')
log(f'Total time {(time.time() - T_START) / 60:.1f} min')
log('ALL DONE')

## Done

Send Claude the text of `MyDrive/rooftop_solar/logs/05_log.txt`.